## **Model Tuning Notebook**

In [ ]:
from numba import cuda
import tensorflow as tf

def clear_gpu_memory():
    device = cuda.get_current_device()
    device.reset() # dump the memory contents to free up the memory (it accumulates over the session)
    
# CUDA (Nvidia GPU Computing)
if len(tf.config.list_physical_devices('GPU')) > 0:
    gpus = tf.config.list_physical_devices('GPU')
    print("Num GPUs Available: ", len(gpus))
    
    clear_gpu_memory()
    
    tf.config.experimental.set_memory_growth(gpus[0], True)

    tf.config.set_logical_device_configuration(
    gpus[0], 
    [tf.config.LogicalDeviceConfiguration(memory_limit=8192)])  # limit to 4GB

    tf.compat.v1.disable_eager_execution()


In [ ]:
# because the utils in the src are designed to be run from the root of the project,
# and by default jupyter runs from the notebook directory we need to change the working directory to the root

import os
import sys
sys.path.append('../')

def find_project_root(filename=".git"): # .git is located in the root of the project
    current_dir = os.getcwd()
    while current_dir != os.path.dirname(current_dir): # stops only when at the root (moves up 1 level each iteration)
        if filename in os.listdir(current_dir):
            return current_dir
        current_dir = os.path.dirname(current_dir)

project_root = find_project_root()
os.chdir(project_root)  # change the working directory to the project root

print("Project root:", project_root, "CWD:", os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

import kerastuner as kt

from src.utils.modeling import *
from src.utils.preproc import *
from src.utils.visualizations import *

from keras.models import Model, Sequential, load_model
from keras.applications import VGG16, InceptionV3
from keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D, Conv2D, MaxPooling2D
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

from sklearn.metrics import classification_report

### **Binary Classification Models Fine-Tuning**

#### **Hyperband Model Tuning**

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
callbacks = [reduce_lr, early_stopping]

tuner = kt.Hyperband(
    keras_tuning_model_builder,
    objective='val_loss',
    max_epochs=10,
    factor=2,
    directory='hyperband_logs',
    project_name='binary_breast_cancer_cnn',
)

train_gen, val_gen, test_gen, class_weights, steps_per_epoch = preproc_pipeline(
    image_resolution=(224, 224),
    classification_type='binary',
    csv_path='image_metadata/updated_image_data.csv',
    use_data_augmentation=False,
    augmented_images_per_image=5,
    batch_size=32
)

tuner.search(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)

In [ ]:
def sequential_binary_model(
    conv_filter_1=64, 
    conv_filter_2=64, 
    conv_filter_3=128, 
    conv_filter_4=128, 
    conv_filter_5=256, 
    dense_units_0=256, 
    dropout_rate_0=0.2, 
    learning_rate=0.0001, 
    input_shape=(224, 224, 3)
):
    model = Sequential()
    model.add(Conv2D(conv_filter_1, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_2, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_3, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_4, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_5, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(dense_units_0, activation='relu'))
    model.add(Dropout(dropout_rate_0))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
train_gen, val_gen, test_gen, class_weights, steps_per_epoch = preproc_pipeline(image_resolution=(224, 224),
                                                    classification_type='binary',
                                                    use_data_augmentation=True,
                                                    augmented_images_per_image=5,
                                                    batch_size=32)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
callbacks = [reduce_lr, early_stopping]

sequential_binary = sequential_binary_model()

fitted_sequential_binary = train_model(train_gen, val_gen, sequential_binary, callbacks=callbacks, class_weights=class_weights, epochs=3, steps_per_epoch=steps_per_epoch)

In [ ]:
save_model(fitted_sequential_binary, 'src/models/VGG16_binary')
save_training_history(fitted_sequential_binary, 'src/models/VGG16_binary')

In [ ]:
fitted_sequential_binary = load_model('src\models\Hyperband_binary\saved_model.h5')
evaluate_model(fitted_sequential_binary, classification_type='binary')

#### **Transfer Learning**

**VGG 16**

In [ ]:
def binary_classification_vgg16_model(input_shape=(224, 224, 3), trainable_layers=10, learning_rate=0.0001):
    
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # train some layers and freeze others
    for layer in base_model.layers[:-trainable_layers]:
        layer.trainable = False
    for layer in base_model.layers[-trainable_layers:]:
        layer.trainable = True

    model = Sequential()
    
    model.add(base_model)
    
    model.add(Flatten())
    
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    
    model.add(Dense(1, activation='sigmoid'))
    
    # low learning rate for fine tuning
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    return model

In [ ]:
train_gen, val_gen, test_gen, class_weights, steps_per_epoch = preproc_pipeline(image_resolution=(224, 224),  
                                                    classification_type='binary',
                                                    use_data_augmentation=True,
                                                    augmented_images_per_image=5,
                                                    batch_size=32)

vgg16_binary = binary_classification_vgg16_model()

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
callbacks = [reduce_lr, early_stopping]
    

fitted_vgg16_binary = train_model(train_gen, val_gen, vgg16_binary, callbacks=callbacks, class_weights=class_weights, epochs=10, steps_per_epoch=steps_per_epoch)

In [ ]:
save_model(fitted_vgg16_binary, 'src/models/VGG16_binary')
save_training_history(fitted_vgg16_binary, 'src/models/VGG16_binary')

In [ ]:
fitted_vgg16_binary = load_model('src\models\VGG16_binary\saved_model.h5')
evaluate_model(fitted_vgg16_binary, classification_type='binary')

### **Multiclass Classification Models Fine-Tuning**

#### **Hyperband Model Tuning**

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=2, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
callbacks = [reduce_lr, early_stopping]

tuner = kt.Hyperband(
    keras_tuning_model_builder,
    objective='val_loss',
    max_epochs=10,
    factor=2,
    directory='hyperband_logs',
    project_name='multiclass_breast_cancer_cnn',
)

train_gen, val_gen, test_gen, class_weights, steps_per_epoch = preproc_pipeline(
    image_resolution=(224, 224),
    classification_type='multiclass',
    csv_path='image_metadata/updated_image_data.csv',
    use_data_augmentation=True,
    augmented_images_per_image=5,
    batch_size=32
)
tuner.search(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)

In [ ]:
def sequential_multiclass_model(
    conv_filter_1=64, 
    conv_filter_2=96, 
    conv_filter_3=192, 
    conv_filter_4=256, 
    conv_filter_5=256, 
    dense_units_0=128, 
    dropout_rate_0=0.2, 
    learning_rate=0.0001, 
    input_shape=(224, 224, 3)
):
    model = Sequential()
    model.add(Conv2D(conv_filter_1, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_2, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_3, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_4, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(conv_filter_5, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(dense_units_0, activation='relu'))
    model.add(Dropout(dropout_rate_0))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
train_gen, val_gen, test_gen, class_weights, steps_per_epoch = preproc_pipeline(image_resolution=(224, 224),
                                                    classification_type='binary',
                                                    use_data_augmentation=True,
                                                    augmented_images_per_image=5,
                                                    batch_size=32)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
callbacks = [reduce_lr, early_stopping]

sequential_multiclass = sequential_multiclass_model()

fitted_sequential_multiclass = train_model(train_gen, val_gen, sequential_multiclass, callbacks=callbacks, class_weights=class_weights, epochs=10, steps_per_epoch=steps_per_epoch)

In [ ]:
save_model(fitted_sequential_multiclass, 'src/models/VGG16_multiclass')
save_training_history(fitted_sequential_multiclass, 'src/models/VGG16_multiclass')

In [ ]:
fitted_sequential_multiclass = load_model('src\models\Hyperband_multiclass\saved_model.h5')
evaluate_model(fitted_sequential_binary, classification_type='multiclass')

#### **Transfer Learning**

**VGG 16**

In [ ]:
def multiclass_classification_vgg16_model(input_shape=(224, 224, 3), trainable_layers=10):
    
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # train some layers and freeze others
    for layer in base_model.layers[:-trainable_layers]:
        layer.trainable = False
    for layer in base_model.layers[-trainable_layers:]:
        layer.trainable = True

    model = Sequential()
    
    model.add(base_model)
    
    model.add(Flatten())
    
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    
    model.add(Dense(8, activation='softmax'))
    
    # low learning rate for fine tuning
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

In [ ]:
train_gen, val_gen, test_gen, class_weights, steps_per_epoch = preproc_pipeline(image_resolution=(224, 224), 
                                                    classification_type='multiclass',
                                                    use_data_augmentation=True,
                                                    augmented_images_per_image=5,
                                                    batch_size=32)
                                                    
vgg16_multiclass = multiclass_classification_vgg16_model()

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, min_lr=1e-7)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
callbacks = [reduce_lr, early_stopping]

fitted_vgg16_multiclass = train_model(train_gen, val_gen, vgg16_multiclass, callbacks=callbacks, class_weights=class_weights, epochs=30, steps_per_epoch=steps_per_epoch)

In [ ]:
save_model(fitted_vgg16_multiclass, 'src/models/VGG16_mutlticlass')
save_training_history(fitted_vgg16_multiclass, 'src/models/VGG16_multiclass')

In [ ]:
fitted_vgg16_multiclass = load_model('src\models\VGG16_multiclass\saved_model.h5')
evaluate_model(fitted_vgg16_multiclass, classification_type='multiclass')

In [ ]:
history_vgg16_binary = {
    'loss': [0.3881, 0.2038, 0.1453, 0.1184, 0.1027, 0.0956, 0.0695, 0.0616, 0.0608, 0.0553, 0.0583, 0.0609, 0.0553, 0.0413, 0.0394],
    'accuracy': [0.8250, 0.9180, 0.9421, 0.9529, 0.9617, 0.9628, 0.9740, 0.9770, 0.9767, 0.9795, 0.9777, 0.9783, 0.9807, 0.9846, 0.9856],
    'val_loss': [0.2230, 0.1898, 0.1615, 0.1241, 0.2154, 0.2113, 0.0979, 0.1304, 0.1548, 0.1999, 0.2550, 0.1098, 0.2333, 0.1204, 0.1753],
    'val_accuracy': [0.9106, 0.9191, 0.9309, 0.9519, 0.9250, 0.9148, 0.9680, 0.9562, 0.9637, 0.9410, 0.9317, 0.9604, 0.9191, 0.9722, 0.9528],
}

In [ ]:
p plt.figure(figsize=(25, 8))
for i, metric in enumerate(['loss', 'accuracy']):
    plt.subplot(1, 2, i + 1)
    plot_results(history_vgg16_multiclass, metric, label='VGG16 Multiclass')
plt.show()

In [ ]:
plt.figure(figsize=(20, 8))

models = [
    ('VGG16 Binary', history_sequential_multiclass),
    ('VGG16 Multiclass', history_vgg16_multiclass)
]

for i, (label, history) in enumerate(models):
    for j, metric in enumerate(['loss', 'accuracy']):
        plt.subplot(1, 2, j + 1)
        plot_results(history, metric, label)

plt.tight_layout()
plt.show()